In [ ]:
!pip install -q langchain==0.2.11 langchain-community==0.2.10 langchain-core==0.2.23 langchain-text-splitters==0.2.2 langchain-huggingface==0.0.3 faiss-cpu pypdf bitsandbytes accelerate transformers sentence-transformers

print("✅ Installation terminée.")
print("⚠️ OBLIGATOIRE : Menu 'Exécution' > 'Redémarrer la session' maintenant.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.3/990.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.2/374.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [ ]:
import torch
import os
from pypdf import PdfReader

# Imports LangChain & FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Imports pour le modèle IA local
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
print("🚀 Démarrage du programme RAG Local (GPU T4)...")

🚀 Démarrage du programme RAG Local (GPU T4)...


# --- 1. CONFIGURATION ET LECTURE DU PDF ---

In [ ]:
chemin_pdf = 'Societe-Generale-Pilier-3_T2-2022_FR.pdf'

if not os.path.exists(chemin_pdf):
    print(f"❌ ERREUR BLOQUANTE : Le fichier '{chemin_pdf}' est introuvable.")
    print("👉 Glisse le fichier PDF dans le dossier 'Fichiers' à gauche de l'écran.")
else:
    print(f"📂 Lecture du fichier PDF...")
    lecteur = PdfReader(chemin_pdf)
    texte_brut = ''
    for page in lecteur.pages:
        t = page.extract_text()
        if t: texte_brut += t

    # DÉCOUPAGE OPTIMISÉ POUR LES TABLEAUX
    # On prend des gros morceaux (1500 caractères) pour ne pas couper les tableaux financiers
    print("✂️  Découpage du texte...")
    decoupeur = CharacterTextSplitter(
        separator="\n",
        chunk_size=1200,
        chunk_overlap=300,
        length_function=len
    )
    textes = decoupeur.split_text(texte_brut)
    print(f"   -> {len(textes)} morceaux générés.")

📂 Lecture du fichier PDF...
✂️  Découpage du texte...
   -> 351 morceaux générés.


# --- 2. EMBEDDINGS (MÉMOIRE) ---

In [ ]:
print("🧠 Création de la base vectorielle (Indexation)...")
# On utilise le GPU (cuda) pour aller vite
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)
docsearch = FAISS.from_texts(textes, embeddings)
print("✅ Base de données prête.")

🧠 Création de la base vectorielle (Indexation)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Base de données prête.


# --- 3. CHARGEMENT DU MODÈLE ZEPHYR 7B ---

In [ ]:
print("🤖 Chargement du modèle Zephyr 7B (Version optimisée 4-bit)...")
print("   ⏳ Patience, cela télécharge environ 5 Go (1 à 2 minutes)...")

# Configuration pour compresser le modèle et le faire tenir dans la mémoire gratuite
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # La magie opère ici (compression)
    device_map="auto"
)

# Création du cerveau de réponse
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1500,      # Longueur max de la réponse
    temperature=0.1,         # 0.1 = Créativité faible (pour être factuel)
    repetition_penalty=1.15, # Pour éviter qu'il répète en boucle
    return_full_text=False   # On veut juste la réponse, pas la question répétée
)

llm_local = HuggingFacePipeline(pipeline=pipe)

🤖 Chargement du modèle Zephyr 7B (Version optimisée 4-bit)...
   ⏳ Patience, cela télécharge environ 5 Go (1 à 2 minutes)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

model-00006-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/816M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0


# --- 4. PROMPT FRANÇAIS ---

In [ ]:
# On donne des ordres stricts au modèle pour qu'il parle français et soit pro
template_francais = """<|system|>
Tu es un expert financier. Tu analyses un rapport bancaire (Pilier 3).
Utilise UNIQUEMENT le contexte ci-dessous pour répondre.
Si la réponse n'est pas dans le texte, dis "Information non trouvée".
Réponds en français de manière synthétique.</s>
<|user|>
Contexte:
{context}

Question: {question}</s>
<|assistant|>"""

PROMPT = PromptTemplate(
    template=template_francais,
    input_variables=["context", "question"]
)

# --- 5. CRÉATION DE LA CHAÎNE ---

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
        llm=llm_local,
        chain_type="stuff",
        retriever=docsearch.as_retriever(search_kwargs={"k": 5}), # On lit 5 pages à la fois pour trouver l'info
        chain_type_kwargs={"prompt": PROMPT}
    )

# --- 6. INTERROGATION (FORMAT MANUEL) ---

In [ ]:
print("\n" + "="*50)
print("      RÉSULTATS DE L'ANALYSE")
print("="*50)


      RÉSULTATS DE L'ANALYSE


# --- REQUÊTE 1 ---

In [ ]:
requete = "quel était le montant des RWA (Expositions pondérées) à la société générale?"
print(f"\n❓ Question : {requete}")
# Note : qa_chain.invoke fait la recherche + la génération en une seule fois
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)


❓ Question : quel était le montant des RWA (Expositions pondérées) à la société générale?
💡 Réponse : Le montant total des RWA (Expositions pondérées) à la société générale est de 121 777 777 Euros au 31 décembre 2021. Cependant, il faut noter que cette information ne figure pas dans le contexte fourni pour répondre à la question. Le contexte indique plutôt que les RWA sont divisées en plusieurs catégories selon leur niveau de risque, et que le groupe a mis en place une stratégie visant à proposer de nouveaux services innovants et à être compétitif face aux nouveaux acteurs de la concurrence, notamment les acteurs non bancaires qui peuvent être avantagés par une réglementation plus souple ou moins restrictive. Les risques environnementaux, sociaux et de gouvernance (ESG), y compris ceux liés au changement climatique, sont également susceptibles d'affecter les activités, les résultats et la situation financière du groupe à court, moyen et long terme. Des investissements complémentaires

# --- REQUÊTE 2 ---

In [ ]:
requete = "Quels sont les principaux indicateurs de performance financière mentionnés dans le document ?"
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)


❓ Question : Quels sont les principaux indicateurs de performance financière mentionnés dans le document ?
💡 Réponse : Les principaux indicateurs de performance financière mentionnés dans le document sont :

1. Valeur comptable brute / montant nominal des expositions faisant l'objet de mesures de restructuration
2. Dépréciations cumulées, variations négatives
3. Provision estimée pour les procédures civiles, administratives, fiscales, pénales ou arbitrales dans lesquelles le groupe est impliqué
4. Conséquences financières des procédures civiles, administratives, fiscales, pénales ou arbitrales dans lesquelles le groupe est impliqué
5. Risques de crédit, de contreiparty et de concentration susceptibles d'avoir un effet défavorable significatif sur l'activité du groupe, sa situation financière et ses résultats
6. Risques environnementaux, sociaux et de gouvernance (ESG), y compris ceux liés au changement climatique, qui peuvent affecter les activités, les résultats et la situation finan

# --- REQUÊTE 3 ---

In [ ]:
requete = "Qui est l'auteur du document? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)


❓ Question : Qui est l'auteur du document? 
💡 Réponse : Le document ne fournit aucune information sur l'identité de l'auteur. Il peut s'agir d'un document généré automatiquement ou d'un document anonyme.
------------------------------


# --- REQUÊTE 4 ---

In [ ]:
requete = "Quels sont les risques mentionnés dans le document? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)


❓ Question : Quels sont les risques mentionnés dans le document? 
💡 Réponse : Les risques mentionnés dans le document sont :

1. Risques environnementaux, sociaux et de gouvernance (ESG) qui peuvent affecter les activités, les résultats et la situation financière du groupe à court et long terme. Ceux-ci incluent les risques liés au changement climatique qui peuvent avoir un impact direct sur l'évolution du climat, les taux d'intérêt, les prix des titres (actions, obligations) et des matières premières, des dérivés et tout autre produit réactif.

2. Risques physiques liés au changement climatique qui peuvent affecter les activités de financement, d'investissement et de services.

3. Risques opérationnels : risque d'une insuffisance ou d'une défaillance des processus, du personnel et des systèmes d'information ou d'événements externes. Ces risques comprennent également les risques de non-conformité, les risques de réputation et les risques de conduite inappropriée (misconduct).

4. Risq

# --- REQUÊTE 5 ---

In [ ]:
requete = "Quel est le montant des fonds propres (Total Capitaux Propres) à la fin de la période de déclaration? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)


❓ Question : Quel est le montant des fonds propres (Total Capitaux Propres) à la fin de la période de déclaration? 
💡 Réponse : Il n'y a pas d'information fournie dans ce contexte pour répondre à cette question. Il est nécessaire de consulter les rapports financiers du groupe pour obtenir cette information.
------------------------------
